# 01 · Reconstrucción de traza — IVR Alkosto

En el pipeline original, todos los pasos de una interacción (navegación,
validaciones de flujo, llamadas a webservice, paso a asesor) llegaban mezclados
en una sola columna (`opcionesnavegaciontrazaopciones`). En los datos actuales,
la vista los separa: `traza_opciones` trae la navegación principal y cada
columna `customN` trae **un solo paso** adicional (validaciones, resultados de
webservice, etc.).

Confirmamos contra el PDF del flujo (`Flujo_actualizado_Alkosto_corte_28-04-26`)
que esos pasos de `customN` **sí son nodos de decisión reales** del árbol (carriles
"API": `¿Error de consulta?`, `¿Estado Fraude?`, `¿Tiene PEA?`, etc.), así que se
integran todos — no se excluyen.

**Entrada:** parquet generado por `00_extraccion.ipynb`.

**Salida:**
- `df_pasos_<periodo>.parquet` — formato largo (un paso por fila), ya con el orden
  cronológico correcto. Este es el insumo directo del Notebook 3 (reemplaza el
  `split('|')` + `stack()` que hacía `01_Alk_final.ipynb` sobre la columna cruda).
- `df_traza_completa_<periodo>.parquet` — un renglón por `id_conversacion` con la
  traza reconstruida como string `|codigo;texto;tiempo|...`, solo para auditoría /
  comparación visual contra `traza_opciones` original.

In [ ]:
import warnings
from pathlib import Path

import pandas as pd
from tqdm import tqdm

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)
tqdm.pandas()

### Parámetros — deben coincidir con los usados en `00_extraccion.ipynb`

In [ ]:
FECHA_INICIO = "2026-06-01"
FECHA_FIN = "2026-06-30"

DATA_DIR = Path("../data") if Path("../data").exists() else Path("data")
RAW_DIR = DATA_DIR / "00_raw"
STAGE_DIR = DATA_DIR / "01_staging"
STAGE_DIR.mkdir(parents=True, exist_ok=True)

INPUT_PATH = RAW_DIR / f"df_raw_{FECHA_INICIO}_{FECHA_FIN}.parquet"
OUT_PASOS_PATH = STAGE_DIR / f"df_pasos_{FECHA_INICIO}_{FECHA_FIN}.parquet"
OUT_TRAZA_PATH = STAGE_DIR / f"df_traza_{FECHA_INICIO}_{FECHA_FIN}.parquet"

assert INPUT_PATH.exists(), f"No encuentro {INPUT_PATH} — corre primero 01_extraccion.ipynb"
print(f"Leyendo: {INPUT_PATH}")

Leyendo: data\00_raw\df_raw_2026-06-01_2026-06-30.parquet


In [ ]:
df = pd.read_parquet(INPUT_PATH)
print(df.shape)

ID_COL = "id_conversacion"
COL_TRAZA_PRINCIPAL = "traza_opciones"
columnas_custom = sorted(
    [c for c in df.columns if c.startswith("custom") and c not in ("custom_50",)],
    key=lambda c: int(c.replace("custom", "")) if c.replace("custom", "").isdigit() else 999,
)
print("Columnas fuente que se van a integrar:", [COL_TRAZA_PRINCIPAL] + columnas_custom)

(105499, 42)
Columnas fuente que se van a integrar: ['traza_opciones', 'custom2', 'custom3', 'custom11', 'custom15', 'custom16', 'custom21', 'custom22', 'custom24', 'custom25', 'custom26', 'custom28', 'custom29', 'custom30', 'custom31', 'custom32', 'custom33', 'custom34', 'custom47', 'custom48']


In [ ]:
def separar_traza(traza):
    if pd.isna(traza):
        return []

    pasos = []

    # Elimina el primer "|" si existe
    for paso in str(traza).strip("|").split("|"):
        partes = paso.split(";")

        # Tomar el nombre del nodo (segunda posición)
        if len(partes) >= 2:
            pasos.append(partes[1].strip())
        else:
            pasos.append(None)

    return pasos

# Expandir la traza
df_traza = df.copy()

df_traza = df_traza.join(
    df_traza["traza_opciones"].apply(separar_traza).apply(pd.Series)
)

# Renombrar columnas
columnas = [c for c in df_traza.columns if isinstance(c, int)]

df_traza.rename(
    columns={i: f"traza_{i+1}" for i in columnas},
    inplace=True
)

columnas_traza = [c for c in df_traza.columns if c.startswith("traza_")]

print(df_traza.shape)
print(f"Número de pasos máximos: {len(columnas_traza)}")

df_traza[["id_conversacion"] + columnas_traza]
df_traza.head(3)

(105499, 86)
Número de pasos máximos: 45


,organizacion,id_conversacion,ani,dnis,division,nombre_ivr,fecha_hora_ingreso,fecha_hora_fin,duracion_ivr,duracion_navegacion,duracion_transaccional,duracion_paso_ce,duracion_desborde,traza_opciones,ultima_opcion,tipo_ult_opcion,paso_ce,tipo_desconexion,id_campana,atributos_custom,custom_50,fecha_inicio,fecha_fin,custom2,custom16,custom21,custom22,custom28,custom29,custom30,custom3,custom11,custom15,custom24,custom25,custom32,custom34,custom47,custom31,custom33,custom26,custom48,traza_1,traza_2,traza_3,traza_4,traza_5,traza_6,traza_7,traza_8,traza_9,traza_10,traza_11,traza_12,traza_13,traza_14,traza_15,traza_16,traza_17,traza_18,traza_19,traza_20,traza_21,traza_22,traza_23,traza_24,traza_25,traza_26,traza_27,traza_28,traza_29,traza_30,traza_31,traza_32,traza_33,traza_34,traza_35,traza_36,traza_37,traza_38,traza_39,traza_40,traza_41,traza_42,traza_43,traza_44
0,emtelcosas,891f883b-e8c0-4a43-91a8-00f93459a745,+573212202672,+576014073033,Alkosto,"ALK_IVR_PRINCIPAL,Default In-Queue Flow Alkost...",2026-06-01 18:51:32.530,2026-06-01 19:03:54.720,185663,185095.0,1496.0,284.0,0.0,|0;Inicio IVR ;0|3;Habeas data positivo;48030|...,1003;Finalización por fin de flujo;31,None,SI,System,None,{'custom2': '|2;Numero documento ingresado;102...,None,2026-06-01,2026-06-01,|2;Numero documento ingresado;1023941473,|12;Usuario_Identificado_Con_ANI;NO,|101;Consulta ws ActualizaHabeasData;FAILURE,|100;Consulta ws ConsultaHabeasData;FAILURE,|107;Consulta ws CheckAftersalesCases;OK,|108;Consulta ws GetClientByAni;NOK,,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Inicio IVR,Habeas data positivo,Menu principal,Garantias y devoluciones,Repetir informacion,Repeat,Iniciar_Tu_Garantia,Igual_O_Menor_30_Dias,Producto_Deteriorado,Gran_Tamano,Paso agente garantias,Bienvenida encuesta SAC,Primera pregunta SAC,Segunda pregunta SAC,Pasa a buzon = NO,Finalización por fin de flujo,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,emtelcosas,a474401c-9cf1-491b-a677-0ac154ac6141,+573156554180,+576014073033,Alkosto,ALK_IVR_PRINCIPAL,2026-06-01 15:31:24.227,2026-06-01 15:32:22.083,57833,57301.0,1301.0,266.0,0.0,|0;Inicio IVR ;0|999;No input;14897,999;No input;14897,None,NO,External,None,{'custom2': '|2;Numero documento ingresado;100...,None,2026-06-01,2026-06-01,|2;Numero documento ingresado;1007469531,|12;Usuario_Identificado_Con_ANI;NO,NaN,|100;Consulta ws ConsultaHabeasData;FAILURE,NaN,|108;Consulta ws GetClientByAni;NOK,,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Inicio IVR,No input,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,emtelcosas,352a5eb7-1bfd-484f-820e-12503f2f8c56,+573146826588,+576014073033,Alkosto,ALK_IVR_PRINCIPAL,2026-06-01 12:23:54.010,2026-06-01 12:34:28.390,36453,34771.0,1682.0,NaN,0.0,|0;Inicio IVR ;0|20;Usuario_Identificado_Con_A...,9;Transferencia Alkosto - Tuya;14753,None,NO,System,None,"{'custom2': None, 'custom16': '|12;Usuario_Ide...",None,2026-06-01,2026-06-01,NaN,|12;Usuario_Identificado_Con_ANI;SI,NaN,NaN,NaN,|108;Consulta ws GetClientByAni;OK,,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Inicio IVR,Usuario_Identificado_Con_Ani,Solicitud_Para_mi,Menu principal,Transferencia Alkosto - Tuya,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
# --- Extraer columnas de negocio desde customN, y limpiar columnas administrativas ---

def extraer_valor_evento(campo):
    """De un campo tipo '|12;Usuario_Identificado_Con_ANI;NO' devuelve solo el valor final (NO)."""
    if pd.isna(campo):
        return None
    partes = str(campo).strip("|").split(";")
    return partes[-1].strip() if len(partes) >= 2 else None

# Mapa código->nombre de negocio para las columnas que identificamos con certeza a partir del
# PDF de flujo y de los datos de muestra. 
mapa_columnas_negocio = {
    "custom2":  "documento_ingresado",              # ALTA -- confirmado en muestra
    "custom16": "usuario_identificado_ani",          # ALTA -- confirmado en muestra
    "custom21": "ws_actualiza_habeas_data",          # ALTA -- confirmado en muestra
    "custom22": "ws_consulta_habeas_data",           # ALTA -- confirmado en muestra
    "custom28": "ws_check_aftersales_cases",         # ALTA -- confirmado en muestra
    "custom29": "ws_get_client_by_ani",              # ALTA -- confirmado en muestra
    "custom11": "autorizacion_tramite",              # ALTA -- valores SI/NO/SIN AUTORIZACION(ÓN)
    "custom15": "numero_guia_transportadora",        # ALTA -- patrón letra+13 dígitos, típico de guía
    "custom30": "destino_transferencia",             # ALTA -- valores 'Directa_SAMSUNG', 'Asistida_POSVENTA', etc.
    "custom31": "numero_destino_transferencia",      # ALTA -- números telefónicos + 'us.invalid' (error SIP típico)
    "custom47": "marca_producto",                    # ALTA -- valores 'samsung','lg','mabe', etc.
    "custom48": "producto_texto_libre",              # MEDIA -- texto libre tipo 'TCL.', 'Xiaomi.', probablemente dictado por voz
    "custom32": "verificacion_producto_ws",          # MEDIA -- mismo conteo exacto que custom33 (584/233) con OK/FAILURE
    "custom33": "verificacion_producto_flag",        # MEDIA -- versión SI/NO de custom32, mismo conteo -> parecen la misma verificación
    "custom3":  "numero_pedido_ingresado",           # BAJA -- numérico, alta cardinalidad, valores vacíos y '#' (típico de captura DTMF)
    "custom24": "ws_resultado_verificacion_1",       # BAJA -- OK/NOK/FAILURE, no identificado con certeza en el PDF revisado
    "custom25": "ws_resultado_verificacion_2",       # BAJA -- mayoritariamente OK, poca muestra de fallos
    "custom26": "ws_resultado_verificacion_3",       # BAJA -- muy poco frecuente (158 de 105k), casi siempre OK
    "custom34": "ws_resultado_verificacion_frecuente", # BAJA -- es la columna custom más poblada (39.7k), significado sin confirmar
}

for col_origen, col_destino in mapa_columnas_negocio.items():
    if col_origen in df_traza.columns:
        df_traza[col_destino] = df_traza[col_origen].apply(extraer_valor_evento)

# Columnas administrativas / crudas que ya no aportan una vez extraído lo anterior
columnas_a_eliminar = ["organizacion", "division", "atributos_custom", "tipo_ult_opcion", "id_campana", "custom_50"] + list(mapa_columnas_negocio.keys())
columnas_a_eliminar = [c for c in columnas_a_eliminar if c in df_traza.columns]
df_traza = df_traza.drop(columns=columnas_a_eliminar)

print(f"Columnas de negocio extraídas ({len(mapa_columnas_negocio)}):", list(mapa_columnas_negocio.values()))
print(f"Columnas eliminadas ({len(columnas_a_eliminar)}):", columnas_a_eliminar)
print(f"Shape resultante: {df_traza.shape}")

Columnas de negocio extraídas (19): ['documento_ingresado', 'usuario_identificado_ani', 'ws_actualiza_habeas_data', 'ws_consulta_habeas_data', 'ws_check_aftersales_cases', 'ws_get_client_by_ani', 'autorizacion_tramite', 'numero_guia_transportadora', 'destino_transferencia', 'numero_destino_transferencia', 'marca_producto', 'producto_texto_libre', 'verificacion_producto_ws', 'verificacion_producto_flag', 'numero_pedido_ingresado', 'ws_resultado_verificacion_1', 'ws_resultado_verificacion_2', 'ws_resultado_verificacion_3', 'ws_resultado_verificacion_frecuente']
Columnas eliminadas (25): ['organizacion', 'division', 'atributos_custom', 'tipo_ult_opcion', 'id_campana', 'custom_50', 'custom2', 'custom16', 'custom21', 'custom22', 'custom28', 'custom29', 'custom11', 'custom15', 'custom30', 'custom31', 'custom47', 'custom48', 'custom32', 'custom33', 'custom3', 'custom24', 'custom25', 'custom26', 'custom34']
Shape resultante: (105499, 80)


In [ ]:
df_traza.head(3)

,id_conversacion,ani,dnis,nombre_ivr,fecha_hora_ingreso,fecha_hora_fin,duracion_ivr,duracion_navegacion,duracion_transaccional,duracion_paso_ce,duracion_desborde,traza_opciones,ultima_opcion,paso_ce,tipo_desconexion,fecha_inicio,fecha_fin,traza_1,traza_2,traza_3,traza_4,traza_5,traza_6,traza_7,traza_8,traza_9,traza_10,traza_11,traza_12,traza_13,traza_14,traza_15,traza_16,traza_17,traza_18,traza_19,traza_20,traza_21,traza_22,traza_23,traza_24,traza_25,traza_26,traza_27,traza_28,traza_29,traza_30,traza_31,traza_32,traza_33,traza_34,traza_35,traza_36,traza_37,traza_38,traza_39,traza_40,traza_41,traza_42,traza_43,traza_44,documento_ingresado,usuario_identificado_ani,ws_actualiza_habeas_data,ws_consulta_habeas_data,ws_check_aftersales_cases,ws_get_client_by_ani,autorizacion_tramite,numero_guia_transportadora,destino_transferencia,numero_destino_transferencia,marca_producto,producto_texto_libre,verificacion_producto_ws,verificacion_producto_flag,numero_pedido_ingresado,ws_resultado_verificacion_1,ws_resultado_verificacion_2,ws_resultado_verificacion_3,ws_resultado_verificacion_frecuente
0,891f883b-e8c0-4a43-91a8-00f93459a745,+573212202672,+576014073033,"ALK_IVR_PRINCIPAL,Default In-Queue Flow Alkost...",2026-06-01 18:51:32.530,2026-06-01 19:03:54.720,185663,185095.0,1496.0,284.0,0.0,|0;Inicio IVR ;0|3;Habeas data positivo;48030|...,1003;Finalización por fin de flujo;31,SI,System,2026-06-01,2026-06-01,Inicio IVR,Habeas data positivo,Menu principal,Garantias y devoluciones,Repetir informacion,Repeat,Iniciar_Tu_Garantia,Igual_O_Menor_30_Dias,Producto_Deteriorado,Gran_Tamano,Paso agente garantias,Bienvenida encuesta SAC,Primera pregunta SAC,Segunda pregunta SAC,Pasa a buzon = NO,Finalización por fin de flujo,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1023941473,NO,FAILURE,FAILURE,OK,NOK,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,a474401c-9cf1-491b-a677-0ac154ac6141,+573156554180,+576014073033,ALK_IVR_PRINCIPAL,2026-06-01 15:31:24.227,2026-06-01 15:32:22.083,57833,57301.0,1301.0,266.0,0.0,|0;Inicio IVR ;0|999;No input;14897,999;No input;14897,NO,External,2026-06-01,2026-06-01,Inicio IVR,No input,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1007469531,NO,NaN,FAILURE,NaN,NOK,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,352a5eb7-1bfd-484f-820e-12503f2f8c56,+573146826588,+576014073033,ALK_IVR_PRINCIPAL,2026-06-01 12:23:54.010,2026-06-01 12:34:28.390,36453,34771.0,1682.0,NaN,0.0,|0;Inicio IVR ;0|20;Usuario_Identificado_Con_A...,9;Transferencia Alkosto - Tuya;14753,NO,System,2026-06-01,2026-06-01,Inicio IVR,Usuario_Identificado_Con_Ani,Solicitud_Para_mi,Menu principal,Transferencia Alkosto - Tuya,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,SI,NaN,NaN,NaN,OK,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
#df_traza["custom48_valor"].value_counts(dropna=False)

In [ ]:
# # --- Tiempo de navegación vs. tiempo técnico, y banderas de negocio a partir de customN ---

# # La fuente ya separa navegación de lo técnico -- no hace falta reconstruirlo desde customN:
# #   duracion_ivr            -> duración total del tramo IVR
# #   duracion_navegacion     -> tiempo real que el cliente pasó navegando el menú
# #   duracion_transaccional  -> tiempo en pasos transaccionales (ej. consultas, webservices)
# #   duracion_paso_ce        -> tiempo específico del paso a asesor (Contact Experience)
# #   duracion_desborde       -> tiempo en desborde
# df_traza["duracion_tecnica_ms"] = (
#     df_traza[["duracion_transaccional", "duracion_paso_ce", "duracion_desborde"]]
#     .fillna(0)
#     .sum(axis=1)
# )
# df_traza["pct_tiempo_navegacion"] = (df_traza["duracion_navegacion"] / df_traza["duracion_ivr"] * 100).round(1)
# df_traza["pct_tiempo_tecnico"] = (df_traza["duracion_tecnica_ms"] / df_traza["duracion_ivr"] * 100).round(1)

# # Banderas de negocio a partir de las columnas ws_* ya extraídas de customN
# columnas_ws = [
#     "ws_actualiza_habeas_data", "ws_consulta_habeas_data",
#     "ws_check_aftersales_cases", "ws_get_client_by_ani",
# ]
# columnas_ws_presentes = [c for c in columnas_ws if c in df_traza.columns]

# df_traza["n_ws_llamadas"] = df_traza[columnas_ws_presentes].notna().sum(axis=1)
# df_traza["ws_algun_error"] = df_traza[columnas_ws_presentes].apply(
#     lambda fila: any(str(v).upper() in ("FAILURE", "NOK") for v in fila if pd.notna(v)), axis=1
# )

# print(df_traza[["duracion_ivr", "duracion_navegacion", "pct_tiempo_navegacion", "pct_tiempo_tecnico", "n_ws_llamadas", "ws_algun_error"]].describe(include="all"))

In [ ]:
# --- df_pasos: una fila por paso de navegación (id_conversacion, bloque, op_num, op_text, op_tiempo) ---

def parsear_pasos(traza):
    """De '|0;Inicio IVR ;0|3;Habeas data positivo;48030|...' devuelve una lista de dicts
    con op_num, op_text, op_tiempo para cada paso, en el orden en que aparecen."""
    if pd.isna(traza):
        return []
    pasos = []
    for seg in str(traza).strip("|").split("|"):
        partes = seg.split(";")
        pasos.append({
            "op_num": partes[0].strip() if len(partes) >= 1 else None,
            "op_text": partes[1].strip() if len(partes) >= 2 else None,
            "op_tiempo": partes[2].strip() if len(partes) >= 3 else None,
        })
    return pasos

registros = []
for id_conv, traza in tqdm(
    zip(df["id_conversacion"], df["traza_opciones"]), total=len(df), desc="Parseando trazas"
):
    for bloque, paso in enumerate(parsear_pasos(traza)):
        registros.append({"id_conversacion": id_conv, "bloque": bloque, **paso})

df_pasos = pd.DataFrame(registros)
df_pasos["op_tiempo"] = pd.to_numeric(df_pasos["op_tiempo"], errors="coerce")

print(f"df_pasos: {df_pasos.shape[0]:,} pasos de {df_pasos['id_conversacion'].nunique():,} conversaciones")
df_pasos.head(10)

Parseando trazas: 100%|██████████| 105499/105499 [00:03<00:00, 28290.23it/s]


df_pasos: 818,010 pasos de 97,151 conversaciones


,id_conversacion,bloque,op_num,op_text,op_tiempo
0,891f883b-e8c0-4a43-91a8-00f93459a745,0,0,Inicio IVR,0
1,891f883b-e8c0-4a43-91a8-00f93459a745,1,3,Habeas data positivo,48030
2,891f883b-e8c0-4a43-91a8-00f93459a745,2,17,Menu principal,346
3,891f883b-e8c0-4a43-91a8-00f93459a745,3,7,Garantias y devoluciones,24254
4,891f883b-e8c0-4a43-91a8-00f93459a745,4,14,Repetir informacion,26061
5,891f883b-e8c0-4a43-91a8-00f93459a745,5,997,Repeat,23
6,891f883b-e8c0-4a43-91a8-00f93459a745,6,523,Iniciar_Tu_Garantia,9781
7,891f883b-e8c0-4a43-91a8-00f93459a745,7,527,Igual_O_Menor_30_Dias,8972
8,891f883b-e8c0-4a43-91a8-00f93459a745,8,540,Producto_Deteriorado,5824
9,891f883b-e8c0-4a43-91a8-00f93459a745,9,528,Gran_Tamano,14839


In [ ]:
# # --- Chequeo rápido: ¿el parser de df_pasos reconstruye exactamente traza_opciones? ---
# # Esto NO genera un archivo nuevo -- es solo una prueba de que parsear_pasos() no perdió ni
# # corrompió información. Se corre sobre una muestra para que sea rápido.
# #
# # Nota: algunas conversaciones tienen traza_opciones nulo (no navegaron nada), así que no
# # generan filas en df_pasos. Eso es esperado -- se tratan como traza vacía, no como error.

# muestra_ids = df["id_conversacion"].sample(min(2000, len(df)), random_state=42)

# def reconstruir(grupo):
#     grupo = grupo.sort_values("bloque")
#     return "".join(
#         f"|{row.op_num};{row.op_text};{'' if pd.isna(row.op_tiempo) else int(row.op_tiempo)}"
#         for row in grupo.itertuples()
#     )

# reconstruido = (
#     df_pasos[df_pasos["id_conversacion"].isin(muestra_ids)]
#     .groupby("id_conversacion")
#     .apply(reconstruir, include_groups=False)
#     .reindex(muestra_ids, fill_value="")  # conversaciones sin pasos -> traza vacía, no NaN
# )

# # Comparamos contra una versión de traza_opciones normalizada de la MISMA forma (quitando
# # espacios en cada texto), para que la comparación sea justa y no la ensucie un espacio suelto
# def normalizar(traza):
#     pasos = parsear_pasos(traza)
#     if not pasos:
#         return ""
#     return reconstruir(pd.DataFrame(pasos).assign(bloque=range(len(pasos))))

# original_normalizado = df.set_index("id_conversacion").loc[muestra_ids, "traza_opciones"].apply(normalizar)
# original_normalizado.index = muestra_ids  # por si hubiera ids duplicados en el índice tras el .loc

# coincide = (reconstruido.values == original_normalizado.values)
# print(f"Coinciden {coincide.sum()} de {len(coincide)} ({coincide.mean()*100:.1f}%)")
# if not coincide.all():
#     print("\nEjemplos que no coinciden (revisar):")
#     display(pd.DataFrame({
#         "id_conversacion": muestra_ids.values,
#         "reconstruido": reconstruido.values,
#         "original_normalizado": original_normalizado.values,
#     })[~coincide].head(3))

In [ ]:
# def parsear_pasos(valor):
#     """Convierte '|cod;texto;tiempo|cod;texto;tiempo' en una lista de tuplas
#     (codigo, texto, tiempo). Ignora vacíos y fragmentos mal formados (no 3 partes).
#     Devuelve también el conteo de fragmentos descartados por mal formados.
#     """
#     if valor is None or (isinstance(valor, float) and pd.isna(valor)):
#         return [], 0
#     partes = str(valor).split("|")
#     pasos = []
#     descartados = 0
#     for parte in partes:
#         parte = parte.strip()
#         if not parte:
#             continue
#         campos = parte.split(";")
#         if len(campos) != 3:
#             descartados += 1
#             continue
#         codigo, texto, tiempo = (c.strip() for c in campos)
#         pasos.append((codigo, texto, tiempo))
#     return pasos, descartados

### Exportar

In [ ]:
df_pasos.to_parquet(OUT_PASOS_PATH, index=False)
df_traza.to_parquet(OUT_TRAZA_PATH, index=False)

print("Guardado:")
print(f"  {OUT_PASOS_PATH}    -> {df_pasos.shape}  (largo, insumo del grafo)")
print(f"  {OUT_TRAZA_PATH} -> {df_traza.shape}  (una fila por conversación: métricas de tiempo y banderas de negocio)")

Guardado:
  data\01_staging\df_pasos_2026-06-01_2026-06-30.parquet    -> (818010, 5)  (largo, insumo del grafo)
  data\01_staging\df_traza_2026-06-01_2026-06-30.parquet -> (105499, 80)  (una fila por conversación: métricas de tiempo y banderas de negocio)


In [ ]:
# Muestra Excel para revisión manual
muestra_path = STAGE_DIR / f"df_traza_muestra_{FECHA_INICIO}_{FECHA_FIN}.xlsx"
df_traza.head(500).to_excel(muestra_path, index=False)
print(f"Muestra Excel: {muestra_path}")

# Muestra Excel para revisión manual
muestra_path = STAGE_DIR / f"df_pasos_muestra_{FECHA_INICIO}_{FECHA_FIN}.xlsx"
df_pasos.head(500).to_excel(muestra_path, index=False)
print(f"Muestra Excel: {muestra_path}")

Muestra Excel: data\01_staging\df_traza_muestra_2026-06-01_2026-06-30.xlsx
Muestra Excel: data\01_staging\df_pasos_muestra_2026-06-01_2026-06-30.xlsx
